In [1]:
import kagglehub

# Download latest version
path1 = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")

print("Path to dataset files:", path1)

100%|██████████| 63.7k/63.7k [00:00<00:00, 89.4MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/atharvaingle/crop-recommendation-dataset/versions/1


In [2]:
import kagglehub

# Download latest version
path2 = kagglehub.dataset_download("akshatgupta7/crop-yield-in-indian-states-dataset")

print("Path to dataset files:", path2)

100%|██████████| 476k/476k [00:00<00:00, 134MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/akshatgupta7/crop-yield-in-indian-states-dataset/versions/1


**Load and Preview**

In [3]:
import pandas as pd
import os

# 1. Load the Crop Recommendation Dataset
# The kaggle path points to the folder, so we join it with the specific CSV filename
rec_file_path = os.path.join(path1, "Crop_recommendation.csv")
df_recommendation = pd.read_csv(rec_file_path)

print("--- Crop Recommendation Dataset Preview ---")
display(df_recommendation.head())


# 2. Load the Crop Yield Dataset
yield_file_path = os.path.join(path2, "crop_yield.csv")
df_yield = pd.read_csv(yield_file_path)

print("\n--- Crop Yield Dataset Preview ---")
display(df_yield.head())

--- Crop Recommendation Dataset Preview ---


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice



--- Crop Yield Dataset Preview ---


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


**Import Libraries and Prepare the Data:**

First, we need to separate our input features (N, P, K, etc.) from our target variable (the crop name). Machine learning models work best with numbers, so we will also encode the crop names into numeric IDs.

In [5]:
# Install xgboost if not already installed
!pip install xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import joblib # For saving the model later

# Assuming df_recommendation is already loaded from your previous step
# df_recommendation = pd.read_csv(os.path.join(path1, "Crop_recommendation.csv"))

# 1. Separate Features (X) and Target (y)
X = df_recommendation[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df_recommendation['label']

# 2. Encode the target labels (e.g., 'rice' -> 0, 'maize' -> 1)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 3. Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 2.9 MB/s eta 0:00:00
Training data shape: (1760, 7)
Testing data shape: (440, 7)


**Train the XGBoost Model:**

XGBoost is an ensemble learning method that builds multiple decision trees. It is incredibly fast and highly accurate for tabular data like this.

In [6]:
# 4. Initialize the XGBoost Classifier
# We use 'multi:softmax' because we are classifying multiple crop types, not just two.
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    random_state=42
)

# 5. Train the model
print("Training the XGBoost model...")
xgb_model.fit(X_train, y_train)
print("Training complete!")

Training the XGBoost model...
Training complete!


**Evaluate Accuracy:**

Now, we test the model on the 20% of data it has never seen before to prove it meets our high accuracy goal.

In [7]:
# 6. Make predictions on the test set
y_pred = xgb_model.predict(X_test)

# 7. Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy * 100:.2f}%")

# Optional: Print a detailed report showing precision/recall for every single crop
# print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


--- Model Evaluation ---
Accuracy: 98.41%


**Test it with Custom Data:**
Let's see the model in action. We can feed it hypothetical soil and weather parameters typical for the Northern Karnataka region—moderate rainfall, warmer temperatures, and slightly alkaline soil—to see what it recommends.

In [8]:
# 8. Function to make a new prediction
def recommend_crop(n, p, k, temp, humidity, ph, rainfall):
    # Create a dataframe for the input
    input_data = pd.DataFrame([[n, p, k, temp, humidity, ph, rainfall]],
                              columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])

    # Predict the encoded label
    prediction_encoded = xgb_model.predict(input_data)

    # Decode back to the string name
    crop_name = label_encoder.inverse_transform(prediction_encoded)
    return crop_name[0]

# Testing with hypothetical regional conditions:
# e.g., NPK levels of 90/40/40, Temp 28°C, Humidity 60%, pH 7.5, Rainfall 800mm (scaled to ~80mm for dataset consistency)
recommended = recommend_crop(n=90, p=40, k=40, temp=28.5, humidity=60.2, ph=7.5, rainfall=85.5)

print(f"\n--- Custom Prediction ---")
print(f"Based on the input parameters, the recommended crop is: **{recommended.upper()}**")


--- Custom Prediction ---
Based on the input parameters, the recommended crop is: **COFFEE**


**Save the Model for your Web App:**

Once you run this and are happy with the 99%+ accuracy, you need to save the model and the label encoder. You will load these files later into your Django or FastAPI backend so your website can use them.

In [9]:
# 9. Save the trained model and the encoder
joblib.dump(xgb_model, 'xgboost_crop_model.pkl')
joblib.dump(label_encoder, 'crop_label_encoder.pkl')

print("\nModel and encoder saved successfully! You can download them from the Colab file explorer.")


Model and encoder saved successfully! You can download them from the Colab file explorer.


In [11]:
from google.colab import drive
import shutil
import os # Import the os module

# 1. Mount your Google Drive
drive.mount('/content/drive')

# Define the target directory path
drive_models_dir = '/content/drive/MyDrive/AgriBot_Models'

# Create the directory if it doesn't exist
os.makedirs(drive_models_dir, exist_ok=True)
print(f"Ensured directory exists: {drive_models_dir}")

# 2. Copy the files directly into your Drive
shutil.copy('xgboost_crop_model.pkl', os.path.join(drive_models_dir, 'xgboost_crop_model.pkl'))
shutil.copy('crop_label_encoder.pkl', os.path.join(drive_models_dir, 'crop_label_encoder.pkl'))

# Only copy these if they exist (they might be from a later part of the notebook)
if os.path.exists('rf_yield_model.pkl'):
    shutil.copy('rf_yield_model.pkl', os.path.join(drive_models_dir, 'rf_yield_model.pkl'))
if os.path.exists('yield_model_columns.pkl'):
    shutil.copy('yield_model_columns.pkl', os.path.join(drive_models_dir, 'yield_model_columns.pkl'))

print("Models successfully backed up to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured directory exists: /content/drive/MyDrive/AgriBot_Models
Models successfully backed up to Google Drive!
